In [1]:
from pathlib import Path
from datetime import datetime
import time
import re

import pandas as pd
import numpy as np
import yfinance as yf
from tqdm.notebook import tqdm

In [2]:
def find_project_root(start_path: Path) -> Path:
    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / "README.md").exists() and (path / "config.yaml").exists():
            return path

    raise FileNotFoundError("Project root not found. Make sure README.md and config.yaml exist.")


PROJECT_ROOT = find_project_root(Path.cwd())

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INDEX_UNIVERSE_DIR = RAW_DIR / "index_universe"
INDEX_UNIVERSE_DIR.mkdir(parents=True, exist_ok=True)

NSE_INDEX_MASTER_RAW_PATH = INDEX_UNIVERSE_DIR / "nse_indices_master_raw.csv"
NSE_INDEX_MASTER_CLEAN_PATH = INDEX_UNIVERSE_DIR / "nse_indices_master.csv"

YF_INDEX_CANDIDATES_PATH = INDEX_UNIVERSE_DIR / "yfinance_index_candidates.csv"
YF_INDEX_DISCOVERY_REPORT_PATH = INDEX_UNIVERSE_DIR / "index_yfinance_discovery_report.csv"
VALIDATED_YF_INDICES_PATH = INDEX_UNIVERSE_DIR / "validated_yfinance_indices.csv"

print("Project root:", PROJECT_ROOT)
print("Index universe dir:", INDEX_UNIVERSE_DIR)
print("Expected raw NSE index file:", NSE_INDEX_MASTER_RAW_PATH)

Project root: E:\Projects\marketguard-india
Index universe dir: E:\Projects\marketguard-india\data\raw\index_universe
Expected raw NSE index file: E:\Projects\marketguard-india\data\raw\index_universe\nse_indices_master_raw.csv


In [3]:
nse_indices_raw = pd.read_csv(NSE_INDEX_MASTER_RAW_PATH)

print("Shape:", nse_indices_raw.shape)
display(nse_indices_raw.head())
display(nse_indices_raw.tail())
print(nse_indices_raw.columns.tolist())

Shape: (144, 13)


,Index Name,Index Date,Open Index Value,High Index Value,Low Index Value,Closing Index Value,Points Change,Change(%),Volume,Turnover (Rs. Cr.),P/E,P/B,Div Yield
0,Nifty 50,13-07-2026,24039.4,24259.8,24000.2,24211.00,4.10,.02,322678968,27998.31,20.87,3.13,1.23
1,Nifty Next 50,13-07-2026,71819.15,72209.05,71429.6,72111.40,-280.40,-.39,223183267,11255.14,18.78,3.45,1.23
2,Nifty 100,13-07-2026,25075.8,25283.2,25021,25240.80,-14.30,-.06,545862235,39253.44,20.45,3.19,1.23
3,Nifty 200,13-07-2026,13914.25,14023.55,13881.3,14003.10,-6.15,-.04,1374002857,67633.86,21.88,3.43,1.1
4,Nifty 500,13-07-2026,23194.8,23379.35,23135.3,23347.05,-1.35,-.01,2167074827,93173.4,23.05,3.5,1.03


,Index Name,Index Date,Open Index Value,High Index Value,Low Index Value,Closing Index Value,Points Change,Change(%),Volume,Turnover (Rs. Cr.),P/E,P/B,Div Yield
139,NIFTY Alpha Low-Volatility 30,13-07-2026,26761.2,26827.35,26650.95,26805.75,-78.75,-.29,77114892,12191.99,27.09,4.43,1.03
140,Nifty500 Flexicap Quality 30,13-07-2026,9785.5,9918.5,9747.45,9902.30,61.65,.63,102525256,14128.09,33.59,7.29,1.25
141,Nifty Total Market Momentum Quality 50,13-07-2026,43423.85,43518.5,43206.45,43487.25,-140.00,-.32,161137529,10207.6,35.39,9.72,.97
142,Nifty500 Equal Weight,13-07-2026,14887.35,15029.1,14828.55,15013.10,40.05,.27,2167074827,93173.4,28.69,3.81,.79
143,NIFTY50 Equal Weight,13-07-2026,32586.25,32878.85,32517.1,32825.75,37.45,.11,322678968,27998.31,24.17,3.56,1.31


['Index Name', 'Index Date', 'Open Index Value', 'High Index Value', 'Low Index Value', 'Closing Index Value', 'Points Change', 'Change(%)', 'Volume', 'Turnover (Rs. Cr.)', 'P/E', 'P/B', 'Div Yield']


In [4]:
nse_indices = nse_indices_raw.copy()

nse_indices.columns = (
    nse_indices.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("%", "pct", regex=False)
)

nse_indices = nse_indices.rename(columns={
    "index_name": "index_name",
    "index_date": "index_date",
    "open_index_value": "open",
    "high_index_value": "high",
    "low_index_value": "low",
    "closing_index_value": "close",
    "points_change": "points_change",
    "changepct": "change_pct",
    "turnover_(rs_cr)": "turnover_rs_cr",
    "div_yield": "div_yield",
})

nse_indices["index_name"] = nse_indices["index_name"].astype(str).str.strip()
nse_indices["index_name_clean"] = (
    nse_indices["index_name"]
    .str.lower()
    .str.replace("&", "and", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

nse_indices["snapshot_date"] = datetime.today().date().isoformat()
nse_indices["source"] = "NSE index snapshot"

display(nse_indices.head())
print(nse_indices.columns.tolist())
print("Unique index names:", nse_indices["index_name"].nunique())

,index_name,index_date,open,high,low,close,points_change,change(pct),volume,turnover_rs_cr,p_e,p_b,div_yield,index_name_clean,snapshot_date,source
0,Nifty 50,13-07-2026,24039.4,24259.8,24000.2,24211.00,4.10,.02,322678968,27998.31,20.87,3.13,1.23,nifty 50,2026-07-14,NSE index snapshot
1,Nifty Next 50,13-07-2026,71819.15,72209.05,71429.6,72111.40,-280.40,-.39,223183267,11255.14,18.78,3.45,1.23,nifty next 50,2026-07-14,NSE index snapshot
2,Nifty 100,13-07-2026,25075.8,25283.2,25021,25240.80,-14.30,-.06,545862235,39253.44,20.45,3.19,1.23,nifty 100,2026-07-14,NSE index snapshot
3,Nifty 200,13-07-2026,13914.25,14023.55,13881.3,14003.10,-6.15,-.04,1374002857,67633.86,21.88,3.43,1.1,nifty 200,2026-07-14,NSE index snapshot
4,Nifty 500,13-07-2026,23194.8,23379.35,23135.3,23347.05,-1.35,-.01,2167074827,93173.4,23.05,3.5,1.03,nifty 500,2026-07-14,NSE index snapshot


['index_name', 'index_date', 'open', 'high', 'low', 'close', 'points_change', 'change(pct)', 'volume', 'turnover_rs_cr', 'p_e', 'p_b', 'div_yield', 'index_name_clean', 'snapshot_date', 'source']
Unique index names: 144


In [5]:
def classify_index(index_name: str) -> str:
    name = index_name.lower()

    if "vix" in name:
        return "volatility"

    if "g-sec" in name or "bond" in name or "rate index" in name:
        return "fixed_income"

    if any(x in name for x in ["bank", "it", "auto", "fmcg", "pharma", "metal", "energy", "realty",
                               "oil", "gas", "healthcare", "financial", "psu", "private bank",
                               "telecommunications", "capital goods", "consumer", "cement",
                               "chemicals", "insurance", "power", "retail", "housing"]):
        return "sector"

    if any(x in name for x in ["quality", "alpha", "momentum", "value", "low volatility",
                               "equal weight", "multifactor", "enhanced", "esg", "dividend"]):
        return "factor_strategy"

    if any(x in name for x in ["tata group", "mahindra", "aditya birla", "corporate group"]):
        return "corporate_group"

    if any(x in name for x in ["defence", "digital", "mobility", "tourism", "railways",
                               "manufacturing", "rural", "new age", "internet", "ipo"]):
        return "theme"

    if any(x in name for x in ["nifty 50", "nifty 100", "nifty 200", "nifty 500",
                               "midcap", "smallcap", "large", "total market", "microcap"]):
        return "broad_market"

    return "other"


nse_indices["index_category"] = nse_indices["index_name"].apply(classify_index)

display(nse_indices["index_category"].value_counts())
display(nse_indices[["index_name", "index_category"]].head(30))

nse_indices.to_csv(NSE_INDEX_MASTER_CLEAN_PATH, index=False)
print("Saved:", NSE_INDEX_MASTER_CLEAN_PATH)

index_category
sector             61
other              31
factor_strategy    20
broad_market       19
theme               8
corporate_group     4
volatility          1
Name: count, dtype: int64

,index_name,index_category
0,Nifty 50,broad_market
1,Nifty Next 50,other
2,Nifty 100,broad_market
3,Nifty 200,broad_market
4,Nifty 500,broad_market
5,Nifty Midcap 50,broad_market
6,NIFTY Midcap 100,broad_market
7,NIFTY Smallcap 100,broad_market
8,Nifty50 Dividend Points,factor_strategy
9,Nifty Auto,sector


Saved: E:\Projects\marketguard-india\data\raw\index_universe\nse_indices_master.csv


In [6]:
def build_search_queries(index_name: str) -> list[str]:
    name = index_name.strip()

    queries = [
        name,
        f"{name} index",
        f"{name} NSE",
        f"{name} Yahoo Finance",
    ]

    # Useful normalization for Yahoo searches
    if name.lower().startswith("nifty"):
        queries.append(name.replace("Nifty", "NIFTY"))
        queries.append("^" + name.replace(" ", "").upper())

    # Remove duplicates while preserving order
    seen = set()
    clean_queries = []

    for q in queries:
        q = re.sub(r"\s+", " ", q).strip()
        if q.lower() not in seen:
            clean_queries.append(q)
            seen.add(q.lower())

    return clean_queries

In [7]:
def normalize_quote_result(result: dict, index_name: str, search_query: str, source_method: str) -> dict:
    return {
        "index_name": index_name,
        "search_query": search_query,
        "source_method": source_method,
        "candidate_symbol": result.get("symbol"),
        "candidate_shortname": result.get("shortname") or result.get("shortName"),
        "candidate_longname": result.get("longname") or result.get("longName"),
        "candidate_quote_type": result.get("quoteType"),
        "candidate_type_disp": result.get("typeDisp"),
        "candidate_exchange": result.get("exchange"),
        "candidate_exch_disp": result.get("exchDisp"),
        "candidate_market": result.get("market"),
        "raw_result": str(result),
    }


def yahoo_search_quotes(query: str, index_name: str, max_results: int = 10) -> list[dict]:
    try:
        try:
            search_obj = yf.Search(
                query,
                max_results=max_results,
                news_count=0,
                include_research=False,
            )
        except TypeError:
            search_obj = yf.Search(query, max_results=max_results)

        quotes = getattr(search_obj, "quotes", [])

        return [
            normalize_quote_result(q, index_name, query, "Search")
            for q in quotes
        ]

    except Exception as e:
        return [{
            "index_name": index_name,
            "search_query": query,
            "source_method": "Search",
            "candidate_symbol": None,
            "candidate_shortname": None,
            "candidate_longname": None,
            "candidate_quote_type": None,
            "candidate_type_disp": None,
            "candidate_exchange": None,
            "candidate_exch_disp": None,
            "candidate_market": None,
            "raw_result": f"ERROR: {e}",
        }]


def yahoo_lookup_indices(query: str, index_name: str, count: int = 25) -> list[dict]:
    try:
        lookup_obj = yf.Lookup(query)

        try:
            results = lookup_obj.get_index(count=count)
        except Exception:
            results = getattr(lookup_obj, "index", [])

        if results is None:
            results = []

        return [
            normalize_quote_result(q, index_name, query, "Lookup")
            for q in results
        ]

    except Exception as e:
        return [{
            "index_name": index_name,
            "search_query": query,
            "source_method": "Lookup",
            "candidate_symbol": None,
            "candidate_shortname": None,
            "candidate_longname": None,
            "candidate_quote_type": None,
            "candidate_type_disp": None,
            "candidate_exchange": None,
            "candidate_exch_disp": None,
            "candidate_market": None,
            "raw_result": f"ERROR: {e}",
        }]

In [8]:
def text_tokens(text: str) -> set[str]:
    if pd.isna(text) or text is None:
        return set()

    text = str(text).lower()
    text = text.replace("&", "and")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    tokens = set(text.split())

    stop = {"nifty", "index", "indices", "the", "and", "of", "india"}
    return tokens - stop


def score_candidate(row: pd.Series) -> int:
    score = 0

    index_name = str(row["index_name"])
    symbol = str(row.get("candidate_symbol") or "")
    shortname = str(row.get("candidate_shortname") or "")
    longname = str(row.get("candidate_longname") or "")
    quote_type = str(row.get("candidate_quote_type") or "").lower()
    type_disp = str(row.get("candidate_type_disp") or "").lower()
    exchange = str(row.get("candidate_exchange") or "").lower()
    exch_disp = str(row.get("candidate_exch_disp") or "").lower()

    candidate_text = f"{symbol} {shortname} {longname}".lower()

    if symbol.startswith("^"):
        score += 4

    if "index" in quote_type or "index" in type_disp:
        score += 4

    if "nifty" in candidate_text:
        score += 3

    if "nse" in candidate_text or "nse" in exchange or "nse" in exch_disp:
        score += 2

    if "india" in candidate_text:
        score += 1

    index_tokens = text_tokens(index_name)
    candidate_tokens = text_tokens(candidate_text)

    overlap = len(index_tokens.intersection(candidate_tokens))
    score += overlap * 2

    # Penalize obvious non-index matches
    if quote_type in {"equity", "etf", "mutualfund", "cryptocurrency", "currency"}:
        score -= 4

    return score

In [9]:
DISCOVERY_MAX_RESULTS = 10
DISCOVERY_SLEEP_SECONDS = 0.25

candidate_rows = []

for index_name in tqdm(nse_indices["index_name"].dropna().unique(), desc="Discovering Yahoo index candidates"):
    queries = build_search_queries(index_name)

    for query in queries:
        candidate_rows.extend(
            yahoo_search_quotes(query, index_name, max_results=DISCOVERY_MAX_RESULTS)
        )

        candidate_rows.extend(
            yahoo_lookup_indices(query, index_name, count=DISCOVERY_MAX_RESULTS)
        )

        time.sleep(DISCOVERY_SLEEP_SECONDS)

index_candidates = pd.DataFrame(candidate_rows)

# Remove empty candidate rows
index_candidates = index_candidates[
    index_candidates["candidate_symbol"].notna()
].copy()

index_candidates["candidate_score"] = index_candidates.apply(score_candidate, axis=1)

# Deduplicate same candidate per index
index_candidates = (
    index_candidates
    .sort_values(["index_name", "candidate_score"], ascending=[True, False])
    .drop_duplicates(subset=["index_name", "candidate_symbol"], keep="first")
    .reset_index(drop=True)
)

index_candidates.to_csv(YF_INDEX_CANDIDATES_PATH, index=False)

print("Candidate rows:", len(index_candidates))
display(index_candidates.head(30))
print("Saved:", YF_INDEX_CANDIDATES_PATH)

Discovering Yahoo index candidates:   0%|          | 0/144 [00:00<?, ?it/s]

Candidate rows: 360


,index_name,search_query,source_method,candidate_symbol,candidate_shortname,candidate_longname,candidate_quote_type,candidate_type_disp,candidate_exchange,candidate_exch_disp,candidate_market,raw_result,candidate_score
0,India VIX,India VIX,Search,^INDIAVIX,INDIA VIX,INDIA VIX,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'INDIA VIX', ...",13
1,NIFTY Midcap 100,NIFTY Midcap 100,Search,NIFTY_MIDCAP_100.NS,NIFTY MIDCAP 100,NIFTY MIDCAP 100,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY MIDCAP...",13
2,NIFTY Midcap 100,NIFTY Midcap 100,Search,LICNMID100.NS,LICNAMC - LICNMID100,LIC MF Nifty Midcap 100 ETF,EQUITY,Equity,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'LICNAMC - LI...",5
3,NIFTY Midcap 100,NIFTY Midcap 100,Search,MOM100.NS,MOTILAL OS MIDCAP100 ETF,Motilal Oswal Nifty Midcap 100 ETF,EQUITY,Equity,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'MOTILAL OS M...",5
4,NIFTY Midcap 100,NIFTY Midcap 100,Search,0000H0.KS,KODEX India Nifty Midcap 100,Samsung Kodex India Nifty Midcap 100 ETF,ETF,ETF,KSC,Korea,None,"{'exchange': 'KSC', 'shortname': 'KODEX India ...",4
5,NIFTY SME EMERGE,NIFTY SME EMERGE,Search,NIFTY_SME_EMERGE.NS,NIFTY SME EMERGE,NIFTY SME EMERGE,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY SME EM...",13
6,NIFTY Smallcap 100,NIFTY Smallcap 100,Search,SML100CASE.NS,ZERODHAAMC - SML100CASE,Zerodha Nifty Smallcap 100 ETF,EQUITY,Equity,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'ZERODHAAMC -...",5
7,NIFTY100 Alpha 30,NIFTY100 Alpha 30,Search,NIFTY100_ALPHA_30.NS,NIFTY100 ALPHA 30,NIFTY100 ALPHA 30,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY100 ALP...",15
8,NIFTY100 ESG,NIFTY100 ESG,Search,NIFTY100_ESG.NS,NIFTY100 ESG,NIFTY100 ESG,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY100 ESG...",13
9,NIFTY100 ESG,^NIFTY100ESG,Search,NIFTY100ESGSECLDR.NS,NIFTY100ESGSECLDR,NIFTY100ESGSECLDR,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY100ESGS...",9


Saved: E:\Projects\marketguard-india\data\raw\index_universe\yfinance_index_candidates.csv


In [10]:
top_candidates = (
    index_candidates
    .sort_values(["index_name", "candidate_score"], ascending=[True, False])
    .groupby("index_name")
    .head(5)
    .reset_index(drop=True)
)

display(
    top_candidates[
        [
            "index_name",
            "candidate_symbol",
            "candidate_shortname",
            "candidate_longname",
            "candidate_quote_type",
            "candidate_exchange",
            "candidate_score",
            "source_method",
            "search_query",
        ]
    ].head(50)
)

,index_name,candidate_symbol,candidate_shortname,candidate_longname,candidate_quote_type,candidate_exchange,candidate_score,source_method,search_query
0,India VIX,^INDIAVIX,INDIA VIX,INDIA VIX,INDEX,NSI,13,Search,India VIX
1,NIFTY Midcap 100,NIFTY_MIDCAP_100.NS,NIFTY MIDCAP 100,NIFTY MIDCAP 100,INDEX,NSI,13,Search,NIFTY Midcap 100
2,NIFTY Midcap 100,LICNMID100.NS,LICNAMC - LICNMID100,LIC MF Nifty Midcap 100 ETF,EQUITY,NSI,5,Search,NIFTY Midcap 100
3,NIFTY Midcap 100,MOM100.NS,MOTILAL OS MIDCAP100 ETF,Motilal Oswal Nifty Midcap 100 ETF,EQUITY,NSI,5,Search,NIFTY Midcap 100
4,NIFTY Midcap 100,0000H0.KS,KODEX India Nifty Midcap 100,Samsung Kodex India Nifty Midcap 100 ETF,ETF,KSC,4,Search,NIFTY Midcap 100
5,NIFTY SME EMERGE,NIFTY_SME_EMERGE.NS,NIFTY SME EMERGE,NIFTY SME EMERGE,INDEX,NSI,13,Search,NIFTY SME EMERGE
6,NIFTY Smallcap 100,SML100CASE.NS,ZERODHAAMC - SML100CASE,Zerodha Nifty Smallcap 100 ETF,EQUITY,NSI,5,Search,NIFTY Smallcap 100
7,NIFTY100 Alpha 30,NIFTY100_ALPHA_30.NS,NIFTY100 ALPHA 30,NIFTY100 ALPHA 30,INDEX,NSI,15,Search,NIFTY100 Alpha 30
8,NIFTY100 ESG,NIFTY100_ESG.NS,NIFTY100 ESG,NIFTY100 ESG,INDEX,NSI,13,Search,NIFTY100 ESG
9,NIFTY100 ESG,NIFTY100ESGSECLDR.NS,NIFTY100ESGSECLDR,NIFTY100ESGSECLDR,INDEX,NSI,9,Search,^NIFTY100ESG


In [11]:
core_keywords = [
    "Nifty 50", "Nifty 100", "Nifty 500", "Nifty Bank", "Nifty IT",
    "Nifty Auto", "Nifty FMCG", "Nifty Pharma", "Nifty Metal",
    "Nifty Energy", "Nifty Realty", "Nifty Financial Services",
    "India VIX"
]

core_candidate_view = top_candidates[
    top_candidates["index_name"].isin(core_keywords)
].copy()

display(core_candidate_view)

,index_name,search_query,source_method,candidate_symbol,candidate_shortname,candidate_longname,candidate_quote_type,candidate_type_disp,candidate_exchange,candidate_exch_disp,candidate_market,raw_result,candidate_score
0,India VIX,India VIX,Search,^INDIAVIX,INDIA VIX,INDIA VIX,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'INDIA VIX', ...",13
28,Nifty 100,^NIFTY100,Search,LIX15.NS,NIFTY100 LIQ 15,NIFTY100 LIQ 15,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY100 LIQ...",9
29,Nifty 100,^NIFTY100,Search,NIFTY100LOWVOL30.NS,NIFTY100 LOWVOL30,NIFTY100 LOWVOL30,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY100 LOW...",9
30,Nifty 100,^NIFTY100,Search,NIFTY100_EQL_WGT.NS,NIFTY100 EQL WGT,NIFTY100 EQL WGT,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY100 EQL...",9
31,Nifty 100,Nifty 100,Search,LOWVOL1.NS,KOTAKMAMC - KOTAKLOVOL,Kotak Nifty 100 Low Volatality 30 ETF,EQUITY,Equity,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'KOTAKMAMC - ...",3
32,Nifty 100,Nifty 100,Search,TOP100CASE.NS,ZERODHAAMC - TOP100CASE,Zerodha Nifty 100 ETF,EQUITY,Equity,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'ZERODHAAMC -...",3
38,Nifty 50,Nifty 50,Search,^NSEI,NIFTY 50,NIFTY 50,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY 50', '...",15
39,Nifty 50,^NIFTY50,Search,NIFTY500MOMENTM50.NS,NIFTY500MOMENTM50,NIFTY500MOMENTM50,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY500MOME...",9
40,Nifty 50,^NIFTY50,Search,NV20.NS,NIFTY50 VALUE 20,NIFTY50 VALUE 20,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY50 VALU...",9
41,Nifty 50,Nifty 50,Search,NIFTYBEES.NS,NIP IND ETF NIFTY BEES,Nippon India ETF Nifty 50 BeES,EQUITY,Equity,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIP IND ETF ...",4


In [12]:
VALIDATION_START = "2015-01-01"

def validate_candidate_ticker(candidate_symbol: str) -> dict:
    try:
        df = yf.download(
            candidate_symbol,
            start=VALIDATION_START,
            interval="1d",
            auto_adjust=False,
            actions=True,
            progress=False,
            threads=False,
            repair=True,
        )

        if df.empty:
            return {
                "download_status": "FAILED",
                "rows_returned": 0,
                "start_date": None,
                "end_date": None,
                "columns_returned": "",
                "issue": "No data returned",
            }

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

        df = df.reset_index()

        date_col = "Date" if "Date" in df.columns else "Datetime" if "Datetime" in df.columns else None

        if date_col is None:
            return {
                "download_status": "FAILED",
                "rows_returned": len(df),
                "start_date": None,
                "end_date": None,
                "columns_returned": "|".join(df.columns.astype(str)),
                "issue": "No Date/Datetime column",
            }

        required = ["Open", "High", "Low", "Close"]
        missing = [col for col in required if col not in df.columns]

        if missing:
            return {
                "download_status": "FAILED",
                "rows_returned": len(df),
                "start_date": df[date_col].min(),
                "end_date": df[date_col].max(),
                "columns_returned": "|".join(df.columns.astype(str)),
                "issue": f"Missing required columns: {missing}",
            }

        return {
            "download_status": "OK",
            "rows_returned": len(df),
            "start_date": df[date_col].min(),
            "end_date": df[date_col].max(),
            "columns_returned": "|".join(df.columns.astype(str)),
            "issue": "",
        }

    except Exception as e:
        return {
            "download_status": "ERROR",
            "rows_returned": 0,
            "start_date": None,
            "end_date": None,
            "columns_returned": "",
            "issue": str(e),
        }

In [13]:
candidates_to_validate = (
    index_candidates
    .sort_values(["index_name", "candidate_score"], ascending=[True, False])
    .groupby("index_name")
    .head(3)
    .reset_index(drop=True)
)

validation_rows = []

for _, row in tqdm(candidates_to_validate.iterrows(), total=len(candidates_to_validate), desc="Validating candidate downloads"):
    result = validate_candidate_ticker(row["candidate_symbol"])

    validation_rows.append({
        **row.to_dict(),
        **result,
        "validated_at": datetime.now().isoformat(timespec="seconds"),
    })

    time.sleep(0.20)

index_discovery_report = pd.DataFrame(validation_rows)

index_discovery_report.to_csv(YF_INDEX_DISCOVERY_REPORT_PATH, index=False)

display(index_discovery_report["download_status"].value_counts())
display(index_discovery_report.head(30))
print("Saved:", YF_INDEX_DISCOVERY_REPORT_PATH)

Validating candidate downloads:   0%|          | 0/171 [00:00<?, ?it/s]


1 Failed download:
['VALUEAXIS.NS']: possibly delisted; no price data found  (1h 2026-01-13 -> 2026-07-01)

1 Failed download:
['CONSUMAXIS.NS']: possibly delisted; no price data found  (1h 2025-10-20 -> 2026-03-11)


download_status
OK    171
Name: count, dtype: int64

,index_name,search_query,source_method,candidate_symbol,candidate_shortname,candidate_longname,candidate_quote_type,candidate_type_disp,candidate_exchange,candidate_exch_disp,candidate_market,raw_result,candidate_score,download_status,rows_returned,start_date,end_date,columns_returned,issue,validated_at
0,India VIX,India VIX,Search,^INDIAVIX,INDIA VIX,INDIA VIX,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'INDIA VIX', ...",13,OK,2829,2015-01-01,2026-07-14,Date|Adj Close|Close|Dividends|High|Low|Open|R...,,2026-07-14T19:52:43
1,NIFTY Midcap 100,NIFTY Midcap 100,Search,NIFTY_MIDCAP_100.NS,NIFTY MIDCAP 100,NIFTY MIDCAP 100,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY MIDCAP...",13,OK,2838,2015-01-01,2026-07-14,Date|Adj Close|Close|Dividends|High|Low|Open|R...,,2026-07-14T19:52:46
2,NIFTY Midcap 100,NIFTY Midcap 100,Search,LICNMID100.NS,LICNAMC - LICNMID100,LIC MF Nifty Midcap 100 ETF,EQUITY,Equity,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'LICNAMC - LI...",5,OK,593,2024-02-19,2026-07-14,Date|Adj Close|Close|Dividends|High|Low|Open|R...,,2026-07-14T19:52:46
3,NIFTY Midcap 100,NIFTY Midcap 100,Search,MOM100.NS,MOTILAL OS MIDCAP100 ETF,Motilal Oswal Nifty Midcap 100 ETF,EQUITY,Equity,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'MOTILAL OS M...",5,OK,2849,2015-01-01,2026-07-14,Date|Adj Close|Close|Dividends|High|Low|Open|R...,,2026-07-14T19:52:47
4,NIFTY SME EMERGE,NIFTY SME EMERGE,Search,NIFTY_SME_EMERGE.NS,NIFTY SME EMERGE,NIFTY SME EMERGE,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY SME EM...",13,OK,1,2026-07-14,2026-07-14,Date|Adj Close|Close|Dividends|High|Low|Open|R...,,2026-07-14T19:52:48
5,NIFTY Smallcap 100,NIFTY Smallcap 100,Search,SML100CASE.NS,ZERODHAAMC - SML100CASE,Zerodha Nifty Smallcap 100 ETF,EQUITY,Equity,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'ZERODHAAMC -...",5,OK,206,2025-09-15,2026-07-14,Date|Adj Close|Close|Dividends|High|Low|Open|R...,,2026-07-14T19:52:49
6,NIFTY100 Alpha 30,NIFTY100 Alpha 30,Search,NIFTY100_ALPHA_30.NS,NIFTY100 ALPHA 30,NIFTY100 ALPHA 30,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY100 ALP...",15,OK,1,2026-07-14,2026-07-14,Date|Adj Close|Close|Dividends|High|Low|Open|R...,,2026-07-14T19:52:50
7,NIFTY100 ESG,NIFTY100 ESG,Search,NIFTY100_ESG.NS,NIFTY100 ESG,NIFTY100 ESG,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY100 ESG...",13,OK,1,2026-07-14,2026-07-14,Date|Adj Close|Close|Dividends|High|Low|Open|R...,,2026-07-14T19:52:51
8,NIFTY100 ESG,^NIFTY100ESG,Search,NIFTY100ESGSECLDR.NS,NIFTY100ESGSECLDR,NIFTY100ESGSECLDR,INDEX,Index,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'NIFTY100ESGS...",9,OK,1,2026-07-14,2026-07-14,Date|Adj Close|Close|Dividends|High|Low|Open|R...,,2026-07-14T19:52:52
9,NIFTY100 Quality 30,NIFTY100 Quality 30,Search,HDFCQUAL.NS,HDFCAMC - HDFCQUAL,HDFC NIFTY100 Quality 30 ETF,EQUITY,Equity,NSI,NSE,None,"{'exchange': 'NSI', 'shortname': 'HDFCAMC - HD...",7,OK,339,2025-03-03,2026-07-14,Date|Adj Close|Close|Dividends|High|Low|Open|R...,,2026-07-14T19:52:53


Saved: E:\Projects\marketguard-india\data\raw\index_universe\index_yfinance_discovery_report.csv


In [14]:
validated_ok = index_discovery_report[
    index_discovery_report["download_status"] == "OK"
].copy()

validated_ok["rows_returned"] = pd.to_numeric(validated_ok["rows_returned"], errors="coerce").fillna(0)

best_validated_indices = (
    validated_ok
    .sort_values(
        ["index_name", "candidate_score", "rows_returned"],
        ascending=[True, False, False]
    )
    .drop_duplicates(subset=["index_name"], keep="first")
    .reset_index(drop=True)
)

best_validated_indices = best_validated_indices.rename(columns={
    "candidate_symbol": "yf_ticker",
    "candidate_shortname": "yf_shortname",
    "candidate_longname": "yf_longname",
})

best_validated_indices = best_validated_indices.merge(
    nse_indices[["index_name", "index_category"]].drop_duplicates(),
    on="index_name",
    how="left"
)

best_validated_indices.to_csv(VALIDATED_YF_INDICES_PATH, index=False)

print("Validated usable indices:", len(best_validated_indices))
display(best_validated_indices[
    [
        "index_name",
        "index_category",
        "yf_ticker",
        "yf_shortname",
        "yf_longname",
        "candidate_score",
        "rows_returned",
        "start_date",
        "end_date",
    ]
].head(100))

print("Saved:", VALIDATED_YF_INDICES_PATH)

Validated usable indices: 82


,index_name,index_category,yf_ticker,yf_shortname,yf_longname,candidate_score,rows_returned,start_date,end_date
0,India VIX,volatility,^INDIAVIX,INDIA VIX,INDIA VIX,13,2829,2015-01-01,2026-07-14
1,NIFTY Midcap 100,broad_market,NIFTY_MIDCAP_100.NS,NIFTY MIDCAP 100,NIFTY MIDCAP 100,13,2838,2015-01-01,2026-07-14
2,NIFTY SME EMERGE,other,NIFTY_SME_EMERGE.NS,NIFTY SME EMERGE,NIFTY SME EMERGE,13,1,2026-07-14,2026-07-14
3,NIFTY Smallcap 100,broad_market,SML100CASE.NS,ZERODHAAMC - SML100CASE,Zerodha Nifty Smallcap 100 ETF,5,206,2025-09-15,2026-07-14
4,NIFTY100 Alpha 30,factor_strategy,NIFTY100_ALPHA_30.NS,NIFTY100 ALPHA 30,NIFTY100 ALPHA 30,15,1,2026-07-14,2026-07-14
...,...,...,...,...,...,...,...,...,...
77,Nifty500 Flexicap Quality 30,sector,FLEXIADD.NS,DSPAMC - FLEXIADD,Dsp Mutual Fund - Dsp Nifty500 Flexicap Qualit...,9,180,2025-10-17,2026-07-14
78,Nifty500 Multicap 50:25:25,other,MULTICAP.NS,MIRAEAMC - MULTICAP,Mirae Asset Nifty500 Multicap 50:25:25 ETF,9,465,2024-09-02,2026-07-14
79,Nifty500 Multicap Momentum Quality 50,sector,EMULTIMQ.NS,EDELAMC - EMULTIMQ,Edelweiss Nifty500 Multicap Momentum Quality 5...,11,420,2024-11-05,2026-07-14
80,Nifty500 Quality 50,sector,0P0001XBHK.BO,0P0001XBHK.BO,Axis Nifty500 Quality 50 Index Reg Gr,5,200,2025-09-15,2026-07-13


Saved: E:\Projects\marketguard-india\data\raw\index_universe\validated_yfinance_indices.csv


In [15]:
core_index_names = [
    "Nifty 50",
    "Nifty 100",
    "Nifty 500",
    "Nifty Bank",
    "Nifty Financial Services",
    "Nifty IT",
    "Nifty Auto",
    "Nifty FMCG",
    "Nifty Pharma",
    "Nifty Healthcare Index",
    "Nifty Metal",
    "Nifty Energy",
    "Nifty Oil & Gas",
    "Nifty Realty",
    "Nifty PSU Bank",
    "Nifty Private Bank",
    "India VIX",
]

core_discovered = best_validated_indices[
    best_validated_indices["index_name"].isin(core_index_names)
].copy()

display(core_discovered[
    [
        "index_name",
        "index_category",
        "yf_ticker",
        "yf_shortname",
        "rows_returned",
        "start_date",
        "end_date",
    ]
])

,index_name,index_category,yf_ticker,yf_shortname,rows_returned,start_date,end_date
0,India VIX,volatility,^INDIAVIX,INDIA VIX,2829,2015-01-01,2026-07-14
10,Nifty 100,broad_market,LIX15.NS,NIFTY100 LIQ 15,1,2026-07-14,2026-07-14
12,Nifty 50,broad_market,^NSEI,NIFTY 50,2837,2015-01-02,2026-07-14
13,Nifty 500,broad_market,NIFTY500MOMENTM50.NS,NIFTY500MOMENTM50,1,2026-07-14,2026-07-14
15,Nifty Auto,sector,^CNXAUTO,NIFTY AUTO,2830,2015-01-01,2026-07-14
16,Nifty Bank,sector,^NSEBANK,NIFTY BANK,2843,2015-01-01,2026-07-14
23,Nifty Energy,sector,^CNXENERGY,NIFTY ENERGY,2830,2015-01-01,2026-07-14
24,Nifty FMCG,sector,^CNXFMCG,NIFTY FMCG,2829,2015-01-01,2026-07-14
25,Nifty Financial Services,sector,FINIETF.NS,ICICIPRAMC-ICICIFIN,358,2025-02-03,2026-07-14
28,Nifty Healthcare Index,sector,0P0001VFHA.BO,0P0001VFHA.BO,258,2025-06-23,2026-07-13


# Final selected indexs (Manually by hand)

| Purpose     | Index             | yfinance ticker |    Use for v1 |
| ----------- | ----------------- | --------------: | ------------: |
| Market      | Nifty 50          |         `^NSEI` |           Yes |
| Market      | Nifty 100         |       `^CNX100` |           Yes |
| Volatility  | India VIX         |     `^INDIAVIX` |           Yes |
| Banking     | Nifty Bank        |      `^NSEBANK` |           Yes |
| Auto        | Nifty Auto        |      `^CNXAUTO` |           Yes |
| IT          | Nifty IT          |        `^CNXIT` | Yes |
| Energy      | Nifty Energy      |    `^CNXENERGY` |           Yes |
| FMCG        | Nifty FMCG        |      `^CNXFMCG` |           Yes |
| Metal       | Nifty Metal       |     `^CNXMETAL` |           Yes |
| Pharma      | Nifty Pharma      |    `^CNXPHARMA` |           Yes |
| PSU Bank    | Nifty PSU Bank    |   `^CNXPSUBANK` |           Yes |
| Realty      | Nifty Realty      |    `^CNXREALTY` |           Yes |
| Commodities | Nifty Commodities |      `^CNXCMDT` |           Yes |
| Media       | Nifty Media       |     `^CNXMEDIA` |           Yes |
| MNC         | Nifty MNC         |       `^CNXMNC` |           Yes |
| PSE         | Nifty PSE         |       `^CNXPSE` |           Yes |
